### SQL Homework
Use this notebook to answer the questions.
It can be in the same project as the previous homework.  
When you are ready, **upload** to your github repo, and send me the link (just zip a txt file with the repo's address, and upload it as the homework).  
If your repo is private, invite me: balazs.balogh@cubixedu.com.

#### Import the SparkSession, create it then load the taxi data (yellow_tripdata_2024-08.parquet)

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.types as st
import pyspark.sql.functions as sf

In [2]:
spark = (
    SparkSession
    .builder
    .appName("DE - Week 2. Homework")
    .master("local[*]")
    .getOrCreate()
)

In [3]:
df_read_parquet = (
    spark
    .read
    .format("parquet")
    .option("header",True)
    .option("delimiter",",")
    .load("C:\\Users\\win11\\Documents\\DataEngineer_kepzes\\Codes\\src\\pyspark_project\\src\\data\\yellow_tripdata_2024-09.parquet")

)

In [4]:
df_read_parquet.createOrReplaceTempView("taxi_2024_09")

#### 1. What is the total fare amount for all trips?  
Please round the answer to two decimal places.

In [5]:
spark.sql("""
SELECT ROUND(sum(fare_amount),2) as rounded_fare_amount FROM taxi_2024_09
""").show()

+-------------------+
|rounded_fare_amount|
+-------------------+
|      7.267455708E7|
+-------------------+



#### 2. Show the maximum fare amount, minimum fare amount, and average fare amount for each payment type. Order by payment type.
Round where you need to two decimal places.

In [6]:
spark.sql("""
SELECT 
payment_type,
max(fare_amount),
min(fare_amount),
avg(fare_amount)
FROM 
taxi_2024_09
GROUP BY payment_type
ORDER BY payment_type
""").show()

+------------+----------------+----------------+------------------+
|payment_type|max(fare_amount)|min(fare_amount)|  avg(fare_amount)|
+------------+----------------+----------------+------------------+
|           0|          652.45|          -88.31|19.474801428893986|
|           1|           500.0|          -323.0|20.933846455081362|
|           2|          1862.2|          -999.0|19.028459627608253|
|           3|           599.0|          -599.0| 6.018724000473617|
|           4|           999.0|          -999.0|1.3998142072075723|
+------------+----------------+----------------+------------------+



#### 3. For trips with a fare amount greater than 20, what is the total tip amount for each day (based on the tpep_pickup_datetime)?
Round the tip to two decimal places, and order the results from highest total tip amount.  
Hint: Check DATE() function, to convert tpep_pickup_datetime to date, to get only the YYYY-MM-DD.

In [7]:
spark.sql("""
WITH greater_fares AS (
    SELECT *
    FROM taxi_2024_09
    WHERE fare_amount > 20
)
SELECT
    DATE(tpep_pickup_datetime) AS pickup_date,
    ROUND(SUM(tip_amount), 2) AS total_tip_on_that_date
FROM 
    greater_fares
GROUP BY 
    1
ORDER BY 
    total_tip_on_that_date DESC
""").show(10)

+-----------+----------------------+
|pickup_date|total_tip_on_that_date|
+-----------+----------------------+
| 2024-09-26|             288873.04|
| 2024-09-19|             267403.31|
| 2024-09-25|             259181.89|
| 2024-09-12|             252716.63|
| 2024-09-18|             246127.63|
| 2024-09-22|             244425.39|
| 2024-09-05|              243657.2|
| 2024-09-20|             241722.96|
| 2024-09-24|             241222.24|
| 2024-09-23|             239764.14|
+-----------+----------------------+
only showing top 10 rows



#### 4. For each trip, show the fare amount along with a column that indicates if the trip was "expensive" (greater than 30) or "cheap" (less than or equal to 30).
Hint: Use CASE WHEN for deciding on expensive, or cheap.

In [8]:
spark.sql("""
SELECT
fare_amount,
CASE
WHEN fare_amount > 30 THEN 'Expensive'
ELSE 'Cheap'
END AS ExpensiveOrCheap
FROM
taxi_2024_09
""").show()

+-----------+----------------+
|fare_amount|ExpensiveOrCheap|
+-----------+----------------+
|       47.8|       Expensive|
|        5.1|           Cheap|
|       13.5|           Cheap|
|       24.7|           Cheap|
|       17.0|           Cheap|
|        8.6|           Cheap|
|       0.01|           Cheap|
|       44.3|       Expensive|
|        6.5|           Cheap|
|       19.1|           Cheap|
|        3.0|           Cheap|
|        3.0|           Cheap|
|        3.0|           Cheap|
|       12.8|           Cheap|
|       10.0|           Cheap|
|       10.7|           Cheap|
|        6.5|           Cheap|
|       20.5|           Cheap|
|       42.9|       Expensive|
|       12.8|           Cheap|
+-----------+----------------+
only showing top 20 rows



#### 5. Find the first trip (based on tpep_pickup_datetime) for each VendorID and display the fare amount.
Hint: You can use CTE with ROW_NUMBER().

In [15]:
spark.sql("""
with filtered_fares AS (
SELECT
VendorID,
fare_amount,
tpep_pickup_datetime,
ROW_NUMBER() OVER (PARTITION BY VendorID ORDER BY tpep_pickup_datetime ASC) AS row_num
FROM
taxi_2024_09
)
SELECT
VendorID,fare_amount
from filtered_fares
WHERE row_num = 1
""").show()

+--------+-----------+
|VendorID|fare_amount|
+--------+-----------+
|       1|       24.0|
|       2|        7.2|
|       6|        2.9|
+--------+-----------+



#### 7. Calculate the average trip distance for each VendorID, and assign a label of 'Above Average' or 'Below Average' for each trip based on the distance relative to the VendorID’s average trip distance.
Hint: CTE joined back to the main DataFrame.

In [22]:
spark.sql("""
    WITH filtered_fares AS (
        SELECT 
            VendorID, 
            avg(trip_distance) as avg_trip_dist
        FROM taxi_2024_09
        GROUP BY VendorID
    )
    SELECT 
        t.*, 
        CASE 
            WHEN t.fare_amount > f.avg_trip_dist THEN 'Above Average'
            WHEN t.fare_amount = f.avg_trip_dist THEN 'Like Average'
            ELSE 'Below Average'
        END AS compared_to_average_trip_dist
    FROM taxi_2024_09 t
    JOIN filtered_fares f ON t.VendorID = f.VendorID
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|compared_to_average_trip_dist|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------------------------+
|       1| 2024-09-01 00:05:51|  2024-09-01 00:45:03|              1|          9.8|         1|   